In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib;matplotlib.use('Agg')
from xgboost import XGBClassifier
from sklearn.model_selection import(train_test_split,StratifiedKFold,GridSearchCV,cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import(classification_report,roc_auc_score,average_precision_score,make_scorer,
recall_score,precision_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,precision_recall_curve)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib
import os

df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\Employee_Attrition\Data\Attrition_clean.csv")
X = df.drop('Attrition', axis = 1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.2,random_state=42,stratify=y)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
ratio = neg / pos
print(f'/nRation: {ratio:.2f}')

pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('xgb' ,XGBClassifier(
        scale_pos_weight = ratio,
         reg_alpha = 0.1,
         reg_lambda = 1.0,
         random_state = 42,
         eval_metric = 'aucpr',
        n_jobs = -1 
    ))
])


param_grid = {
    'xgb__n_estimators' :[500,1000],
    'xgb__max_depth' : [3,4],
    'xgb__learning_rate' : [0.01,0.02],
    'xgb__subsample' : [0.8],
    'xgb__colsample_bytree' : [0.8],
    'xgb__min_child_weight' : [3,5,7],
    'xgb__gamma' : [0.3,0.5]
}

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state =42)
scoring = make_scorer(f1_score,pos_label=1)

grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    scoring = 'roc_auc',
    cv=cv,
    verbose = 1,
)
print('/nStarting GridSearch')

grid.fit(X_train,y_train)
print("/nBest Params:",grid.best_params_)
print("/Best CV Score:",grid.best_score_)

best_model = grid.best_estimator_
pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
ap = average_precision_score(y_test,y_prob)
print('/Training Complete')
print(classification_report(y_test, pred, target_names =['Retained','Left']))
print(f'\nAUC-ROC :{auc:.4f}')
print(f'\nAvg Precision :{ap:.4f}')

#Threshold Analysis
print("Threshold Tuning")
best_t, best_p, best_r =0.5,0,0
for t in np.arange(0.10,0.50,0.01):
    y_pred_t = (y_prob>t).astype(int)
    p = precision_score(y_test,y_pred_t,pos_label=1)
    r = recall_score(y_test,y_pred_t,pos_label=1)
    auc = roc_auc_score(y_test,y_pred_t)
    ap =average_precision_score(y_test,y_pred_t)
    print(f"t={t:.2f} | P={p:.3f} R={r:.3f} AUC={auc:.4f} AP={ap:.4f}")
    if (p+r)>(best_p +best_r):
        best_t,best_p,best_r =t,p,r
print(f"\nBest Balance Threshold: {best_t:.2f} | P={best_p:.3f} | R={best_r:.3f}")

THRESHOLD= 0.5
y_pred_final=(y_prob > THRESHOLD).astype(int)
print(classification_report(y_test,y_pred_final, target_names=['Retained','Left']))



/nRation: 5.19
/nStarting GridSearch
Fitting 5 folds for each of 48 candidates, totalling 240 fits
/nBest Params: {'xgb__colsample_bytree': 0.8, 'xgb__gamma': 0.3, 'xgb__learning_rate': 0.01, 'xgb__max_depth': 3, 'xgb__min_child_weight': 7, 'xgb__n_estimators': 1000, 'xgb__subsample': 0.8}
/Best CV Score: 0.8156774801177686
/Training Complete
              precision    recall  f1-score   support

    Retained       0.91      0.88      0.89       247
        Left       0.46      0.55      0.50        47

    accuracy                           0.83       294
   macro avg       0.69      0.72      0.70       294
weighted avg       0.84      0.83      0.83       294


AUC-ROC :0.7968

Avg Precision :0.5431
Threshold Tuning
t=0.10 | P=0.189 R=0.915 AUC=0.5830 AP=0.1862
t=0.11 | P=0.199 R=0.915 AUC=0.6072 AP=0.1957
t=0.12 | P=0.206 R=0.915 AUC=0.6214 AP=0.2018
t=0.13 | P=0.210 R=0.915 AUC=0.6295 AP=0.2055
t=0.14 | P=0.224 R=0.915 AUC=0.6558 AP=0.2185
t=0.15 | P=0.225 R=0.872 AUC=0.6507 AP=0.

In [2]:
cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained','Left'])
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix at Threshold = 0.5')
plt.savefig('Attrition_confusion_matrix.png',dpi=120)
plt.show()

precision, recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test,y_prob)
plt.figure()
plt.plot(recall,precision,label=f'XGB AP={ap:.4f}')
plt.axhline(y=y_test.mean(),color='r',linestyle='--', label=f'Random={y_test.mean():.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig('Attrition_pr_curve.png',dpi=120)
plt.show()

import shap
print("Calculating SHAP")
xgb_model = best_model.named_steps['xgb']
X_test_scaled = best_model[:-1].transform(X_test)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(shap_values,X_test_scaled,feature_names=X_test.columns,plot_type="bar",show=False)
plt.title('Top Features Driving Attrition')
plt.tight_layout()
plt.savefig('Attrition_Shap_importances.png',dpi=120)
plt.show()

plt.figure(figsize=(10,10))
X_test_df = pd.DataFrame(X_test_scaled, columns =X.columns)
sample_idx = X_test_df.sample(200,random_state=42).index
X_test_sample =X_test_df.loc[sample_idx]
shap_sample = shap_values[sample_idx]
shap.summary_plot(shap_sample,X_test_sample,plot_type="dot",show=False)
plt.title('Attrition Drivers')
plt.tight_layout()
plt.savefig('Shap_attrition_beeswarm.png',dpi=120,bbox_inches='tight')
print("Done plottig")
joblib.dump(best_model,'attrition_pipe.pkl')
feature_columns = list(X_train.columns)
joblib.dump(feature_columns, 'feature_names.pkl')
joblib.dump(THRESHOLD,'THRESHOLD.pkl')
print('Saved')


C:\Users\niluc\AppData\Local\Temp\ipykernel_23800\41065243.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\niluc\AppData\Local\Temp\ipykernel_23800\41065243.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Calculating SHAP


C:\Users\niluc\AppData\Local\Temp\ipykernel_23800\41065243.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Done plottig
Saved
